# TP1 - Bases de datos primarias e introducción a Biopython

**Bioinformática 2026**

Este notebook integra las respuestas de navegación en NCBI/UniProt y los ejercicios programáticos con Python y Biopython. Para que sea reproducible, las secuencias usadas en los análisis están guardadas localmente en la carpeta `data/`.

## Objetivo 1a - Recorriendo NCBI

En la página principal de NCBI se puede buscar en muchas bases de datos. Algunas bases reconocibles son:

- **PubMed**: publicaciones científicas.
- **Nucleotide / GenBank**: secuencias nucleotídicas.
- **Protein**: secuencias de proteínas.
- También aparecen otras como **Gene**, **Taxonomy**, **Genome**, **Structure** y **OMIM**.

En búsquedas de secuencias, los campos avanzados son importantes porque permiten restringir la búsqueda. En **Nucleotide** nos parecen especialmente relevantes `Organism`, `Sequence Length`, `Molecule Type`, `Feature Key`, `Publication Date`, `Author` y `Source Database`. En **Protein** son relevantes `Organism`, `Sequence Length`, `Molecular Weight`, `Protein Name`, `Gene Name`, `Publication Date` y `Source Database`.

### Ejemplos de búsquedas en NCBI

Consultas hechas el **30/04/2026** con Entrez:

| Búsqueda | Base | Resultado |
|---|---:|---:|
| `kinase AND Trypanosoma cruzi` | Nucleotide | 7438 |
| `kinase AND Trypanosoma cruzi[organism]` | Nucleotide | 4104 |
| `2002[molwt]` | Protein | 1281 |
| `2002:2009[molwt]` | Protein | 29263 |
| `3000:4000[slen]` | Nucleotide | 30605223 |
| `promoter[fkey] AND srcdb genbank[prop]` | Nucleotide | 36333 |

La diferencia entre las dos búsquedas de `kinase` muestra por qué conviene usar tags/campos: sin `[organism]`, NCBI busca los términos en muchos campos; con `[organism]`, restringe el organismo.

Para **Trypanosoma cruzi cruzi**, la clasificación taxonómica es, de forma resumida: Eukaryota; Discoba; Euglenozoa; Kinetoplastea; Trypanosomatidae; Trypanosoma; Trypanosoma cruzi.

### Ejercicios 1a

Consultas hechas el **30/04/2026** con Entrez:

1. Proteínas humanas de entre 50 y 60 aminoácidos publicadas durante 2022:
   - Base: Protein
   - Query: `Homo sapiens[organism] AND 50:60[slen] AND 2022[pdat]`
   - Resultado: **5386** registros.

2. Secuencias genómicas de *Escherichia coli* identificadas como atenuadores:
   - Base: Nucleotide
   - Query: `Escherichia coli[organism] AND biomol genomic[Properties] AND attenuator[fkey]`
   - Resultado: **12** registros.

3. Proteínas que unen penicilina en *Mycobacterium tuberculosis*:
   - Protein: **38423** registros para `penicillin-binding AND Mycobacterium tuberculosis[organism]`.
   - Gene: **276** registros.
   - Nucleotide: **25665** registros.
   - La base Protein es la más directa para buscar productos proteicos, aunque Gene puede ser más útil si se quiere contar genes en vez de secuencias proteicas.

4. Secuencias nucleotídicas depositadas en GenBank por Venter, JC:
   - Base: Nucleotide
   - Query: `Venter JC[Author] AND srcdb genbank[prop]`
   - Resultado: **7635186** registros.

Los conteos pueden cambiar porque las bases de datos se actualizan constantemente.

## Objetivo 1b - Registro de mioglobina en NCBI

Para la mioglobina humana se usó el registro de proteína **NP_976311.1**.

- En formato **GenPept (Full)** aparecen campos como `LOCUS`, `DEFINITION`, `ACCESSION`, `VERSION`, `DBSOURCE`, `KEYWORDS`, `SOURCE`, `ORGANISM`, `REFERENCE`, `COMMENT`, `FEATURES` y `ORIGIN`. Este formato conserva anotaciones y metadatos.
- En formato **FASTA** aparece sólo un encabezado con identificación/descripción y la secuencia. Es más simple y cómodo para análisis computacional.

Respuestas:

1. GenPept contiene metadatos, referencias, anotaciones de features y secuencia. FASTA contiene encabezado + secuencia.
2. La mioglobina NP_976311.1 posee **154 aminoácidos**.
3. El mRNA asociado **NM_203377.1** posee aproximadamente **1170 nucleótidos** según RefSeq.
4. Para el gen humano **MB**, RefSeq lista **5 variantes de splicing**: NM_005368, NM_203377, NM_203378, NM_203379 y NM_203380. Las variantes difieren mayormente en regiones 5' UTR y dan productos proteicos parecidos.
5. El gen **MB** se encuentra en el cromosoma **22**, brazo **q**, banda **22q12.3**. En Genome Data Viewer, los genes vecinos más cercanos son **RBFOX2** (downstream) y, hacia upstream, miembros del clúster de apolipoproteínas **APOL1, APOL2, APOL3, APOL4** junto con **MYH9**.
6. En este repo se guardaron copias locales en FASTA:
   - `data/mioglobina_gen.fasta`
   - `data/mioglobina_mrna.fasta`
   - `data/mioglobina_proteina.fasta`

## Objetivo 1c - UniProt

Para UniProt se usó la entrada **P02144 / MYG_HUMAN**.

1. Modificaciones postraduccionales: se informa **N-acetilglicina** en el extremo N-terminal. También se anotan sitios de unión al grupo hemo.
2. El gen está en el cromosoma **22** y la proteína canónica tiene **154 aminoácidos**. El gen humano de mioglobina tiene una organización clásica de **3 exones y 2 intrones**.
3. Hay estructura experimental resuelta. Un ejemplo asociado al registro es **PDB 3RGK**, estructura cristalográfica de mioglobina humana mutante K45R.
4. La entrada fue integrada en Swiss-Prot/UniProtKB el **21/07/1986**.

## Preparación del entorno de Python

In [ ]:
from pathlib import Path
from collections import Counter
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
from Bio import SwissProt

PROYECTO = Path.cwd()
if not (PROYECTO / "data").exists():
    PROYECTO = PROYECTO.parent

DATA_DIR = PROYECTO / "data"
RESULTADOS_DIR = PROYECTO / "resultados"
RESULTADOS_DIR.mkdir(exist_ok=True)

## Objetivo 2.1 - Secuencias al azar

Generamos ADN y proteínas al azar con un largo definido por el usuario. Para que el resultado sea reproducible, uso una semilla fija.

In [ ]:
NUCLEOTIDOS = list("ACGT")
AMINOACIDOS = list("ACDEFGHIKLMNPQRSTVWY")


def generar_adn(largo, semilla=7):
    generador = random.Random(semilla)
    return "".join(generador.choice(NUCLEOTIDOS) for _ in range(largo))


def generar_proteina(largo, frecuencias=None, semilla=7):
    rng = np.random.default_rng(semilla)
    if frecuencias is None:
        probabilidades = np.repeat(1 / len(AMINOACIDOS), len(AMINOACIDOS))
    else:
        probabilidades = np.array([frecuencias[aa] for aa in AMINOACIDOS], dtype=float)
        probabilidades = probabilidades / probabilidades.sum()
    return "".join(rng.choice(AMINOACIDOS, size=largo, p=probabilidades))


adn_azar = generar_adn(900)
proteina_azar = generar_proteina(300)

print("ADN al azar:", adn_azar[:80] + "...")
print("Proteína al azar:", proteina_azar[:80] + "...")
print("Longitud ADN:", len(adn_azar))
print("Longitud proteína:", len(proteina_azar))

También podemos fijar frecuencias distintas para los 20 aminoácidos. En este ejemplo aumento la probabilidad de glicina (`G`) y alanina (`A`) para mostrar que el generador respeta pesos no uniformes.

In [ ]:
frecuencias_sesgadas = {aa: 1 for aa in AMINOACIDOS}
frecuencias_sesgadas["A"] = 6
frecuencias_sesgadas["G"] = 6
proteina_sesgada = generar_proteina(300, frecuencias=frecuencias_sesgadas)
print(proteina_sesgada[:100] + "...")

## Objetivo 2.2 - Frecuencias en secuencias

In [ ]:
CODIGO_GENETICO = {
    "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    "TAT": "Y", "TAC": "Y", "TAA": "*", "TAG": "*",
    "TGT": "C", "TGC": "C", "TGA": "*", "TGG": "W",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}


def traducir_adn(adn):
    aminoacidos = []
    for posicion in range(0, len(adn) - 2, 3):
        codon = adn[posicion : posicion + 3].upper()
        aminoacidos.append(CODIGO_GENETICO.get(codon, "X"))
    return "".join(aminoacidos)


def frecuencias(secuencia, alfabeto):
    conteos = Counter(secuencia)
    total = sum(conteos.get(letra, 0) for letra in alfabeto)
    return pd.DataFrame(
        [(letra, conteos.get(letra, 0), conteos.get(letra, 0) / total if total else 0) for letra in alfabeto],
        columns=["símbolo", "conteo", "frecuencia"],
    )


def graficar_frecuencias(df, titulo, ruta):
    ax = df.plot(kind="bar", x="símbolo", y="frecuencia", legend=False, figsize=(9, 4), color="#3b73b9")
    ax.set_title(titulo)
    ax.set_ylabel("Frecuencia")
    ax.set_xlabel("Símbolo")
    plt.tight_layout()
    plt.savefig(ruta, dpi=180)
    plt.show()


freq_adn = frecuencias(adn_azar, NUCLEOTIDOS)
freq_prot = frecuencias(proteina_azar, AMINOACIDOS)
traduccion_adn_azar = traducir_adn(adn_azar).replace("*", "")
freq_trad = frecuencias(traduccion_adn_azar, AMINOACIDOS)

freq_adn

In [ ]:
graficar_frecuencias(freq_adn, "Frecuencias en ADN al azar", RESULTADOS_DIR / "nb_frecuencias_adn_azar.png")
graficar_frecuencias(freq_prot, "Frecuencias en proteína al azar", RESULTADOS_DIR / "nb_frecuencias_proteina_azar.png")
graficar_frecuencias(freq_trad, "Aminoácidos al traducir ADN al azar", RESULTADOS_DIR / "nb_frecuencias_traduccion_adn_azar.png")

## Objetivo 3 - Introducción a Biopython

Biopython permite representar secuencias como objetos `Seq`, calcular GC, obtener complementaria, reversa complementaria, transcribir y traducir.

In [ ]:
my_seq = Seq("GATCGATGGGCCTATATAGGATCGAAAATCTAAGCC")
print("Secuencia:", my_seq)
print("Longitud:", len(my_seq))
print("GC (%):", gc_fraction(my_seq) * 100)
print("Complementaria:", my_seq.complement())
print("Reversa complementaria:", my_seq.reverse_complement())
print("Transcripción:", my_seq.transcribe())
print("Traducción:", my_seq.translate())

## Ejercicio 3.1 - Mioglobina con Biopython

Cargo las secuencias locales de mioglobina. La secuencia genómica guardada corresponde a la región codificante simplificada, por eso la traducción directa coincide con la proteína real. En un contig genómico completo habría intrones y habría que ubicar exones/CDS antes de traducir.

In [ ]:
gen_record = SeqIO.read(DATA_DIR / "mioglobina_gen.fasta", "fasta")
mrna_record = SeqIO.read(DATA_DIR / "mioglobina_mrna.fasta", "fasta")
prot_record = SeqIO.read(DATA_DIR / "mioglobina_proteina.fasta", "fasta")

print(gen_record.id, len(gen_record.seq), "nt")
print(mrna_record.id, len(mrna_record.seq), "nt en copia local")
print(prot_record.id, len(prot_record.seq), "aa")
print("GC gen (%):", round(gc_fraction(gen_record.seq) * 100, 2))

In [ ]:
traduccion_directa = gen_record.seq.translate(to_stop=True)
traduccion_reversa = gen_record.seq.reverse_complement().translate(to_stop=True)

print("Proteína real:       ", prot_record.seq[:80])
print("Traducción directa:  ", traduccion_directa[:80])
print("Traducción reversa:  ", traduccion_reversa[:80])
print("Directa coincide con proteína real:", str(traduccion_directa) == str(prot_record.seq))

La traducción directa de la región codificante da la mioglobina esperada. La reversa complementaria no coincide, lo que indica que esta copia de trabajo está en la orientación codificante. Si se trabaja con la región genómica completa del Genome Browser, hay que considerar que el gen puede estar anotado en la hebra complementaria y que hay intrones.

## Ejercicio 3.2 - SeqRecord, FASTA y GenBank

In [ ]:
fasta_record = SeqIO.read(DATA_DIR / "mioglobina_proteina.fasta", "fasta")
genbank_record = SeqIO.read(DATA_DIR / "mioglobina_gen.gb", "genbank")

print("FASTA")
print("id:", fasta_record.id)
print("name:", fasta_record.name)
print("description:", fasta_record.description)
print("longitud:", len(fasta_record.seq))
print("secuencia como string:", str(fasta_record.seq[:30]))

print("\nGenBank")
print("id:", genbank_record.id)
print("name:", genbank_record.name)
print("description:", genbank_record.description)
print("longitud:", len(genbank_record.seq))
print("cantidad de features:", len(genbank_record.features))
for feature in genbank_record.features:
    print(feature.type, feature.location)

La misma secuencia puede leerse desde FASTA o GenBank, pero GenBank conserva anotaciones (`features`) como `source`, `gene` y `CDS`. Para manipular solo la secuencia alcanza con convertir `record.seq` a string usando `str(record.seq)`.

La descarga automática con Entrez se deja implementada en `scripts/descargas_biopython.py`, pero el notebook usa archivos locales para no depender de internet.

### Ejercicio 3.2.b - Descarga con `Entrez.efetch`

El notebook usa archivos locales en `data/` para que sea reproducible sin
internet. La descarga programática equivalente con Biopython es la que
sigue. Si hay conectividad, baja el FASTA desde NCBI y verifica que la
secuencia coincide con la copia local. Si no hay internet, captura el
error y sigue usando los archivos locales.

El script `scripts/descargas_biopython.py` hace lo mismo desde la línea
de comandos.

In [ ]:
from Bio import Entrez

Entrez.email = "tu_email@example.com"  # NCBI requiere un email de contacto

try:
    with Entrez.efetch(db="protein", id="NP_976311.1", rettype="fasta", retmode="text") as handle:
        fasta_descargado = handle.read()
    print(fasta_descargado[:200])
    print("...")
    # Comparación con la copia local
    local = (DATA_DIR / "mioglobina_proteina.fasta").read_text()
    secuencia_local = "".join(local.splitlines()[1:])
    secuencia_remota = "".join(fasta_descargado.splitlines()[1:])
    print("\n¿Coincide la secuencia con la copia local?", secuencia_remota == secuencia_local)
except Exception as error:
    print("Sin acceso a internet o NCBI rechazó la consulta. Se usa la copia local.")
    print(repr(error))

## Ejercicio 3.3 - Proteínas y formato SwissProt

In [ ]:
with open(DATA_DIR / "mioglobina_proteina_uniprot.txt", encoding="utf-8") as handle:
    swiss_record = SwissProt.read(handle)

print("Entry name:", swiss_record.entry_name)
print("Accessions:", swiss_record.accessions)
print("Longitud:", swiss_record.sequence_length)
print("Secuencia FASTA:")
print(f">{swiss_record.accessions[0]} {swiss_record.entry_name}")
print(swiss_record.sequence)

## Objetivo 4.1 - Distribución de aminoácidos desde ADN al azar vs mioglobina

Comparo la distribución de aminoácidos obtenida al traducir ADN al azar contra la proteína real de mioglobina. El ADN al azar produce una distribución influida por el código genético: aminoácidos codificados por más codones tienden a aparecer más.

In [ ]:
freq_mioglobina = frecuencias(str(prot_record.seq), AMINOACIDOS)
comparacion = freq_trad.merge(freq_mioglobina, on="símbolo", suffixes=("_adn_azar", "_mioglobina"))
comparacion[["símbolo", "frecuencia_adn_azar", "frecuencia_mioglobina"]]

In [ ]:
ax = comparacion.plot(
    x="símbolo",
    y=["frecuencia_adn_azar", "frecuencia_mioglobina"],
    kind="bar",
    figsize=(10, 4),
)
ax.set_title("Aminoácidos: ADN al azar traducido vs mioglobina")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.savefig(RESULTADOS_DIR / "nb_comparacion_adn_azar_mioglobina.png", dpi=180)
plt.show()

A medida que aumenta el tamaño de la secuencia al azar, la distribución se estabiliza. En secuencias cortas hay mucho ruido muestral; en secuencias de miles de nucleótidos las frecuencias se acercan más a lo esperable por azar y por degeneración del código genético. Para secuencias reales, también hace falta aumentar el número de genes/proteínas analizados para que la distribución no dependa demasiado de un solo ejemplo.

### Ejercicio 4.1.c - Convergencia con el tamaño de la secuencia

A medida que crece el largo de la secuencia de ADN al azar, la frecuencia
de aminoácidos obtenida al traducirla se acerca a la distribución teórica
derivada del **código genético**: cada aminoácido aparece en proporción
al número de codones que lo codifican (sobre los 61 codones que codifican
aminoácidos). Medimos la convergencia con la **distancia L1** entre la
frecuencia observada y la esperada por código genético.

In [ ]:
# Frecuencia esperada según el código genético: # codones de cada aa / 61
codones_por_aa = Counter(v for v in CODIGO_GENETICO.values() if v != "*")
total_codones = sum(codones_por_aa.values())  # 61
esperado = {aa: codones_por_aa.get(aa, 0) / total_codones for aa in AMINOACIDOS}

largos = [50, 200, 1000, 5000, 20000, 100000]
filas_conv = []
for largo in largos:
    adn_n = generar_adn(largo, semilla=largo)
    aa_n = traducir_adn(adn_n).replace("*", "")
    if not aa_n:
        continue
    conteos = Counter(aa_n)
    total_aa = sum(conteos.values())
    # distancia L1 entre observado y esperado, dividida por 2 -> rango [0, 1]
    distancia = sum(abs(conteos.get(aa, 0) / total_aa - esperado[aa]) for aa in AMINOACIDOS) / 2
    filas_conv.append({"Largo de ADN (nt)": largo, "Aminoácidos obtenidos": total_aa, "Distancia L1": distancia})

df_conv = pd.DataFrame(filas_conv)

plt.figure(figsize=(8, 4))
plt.plot(df_conv["Largo de ADN (nt)"], df_conv["Distancia L1"], marker="o")
plt.xscale("log")
plt.yscale("log")
plt.title("Convergencia de la distribución de aminoácidos al esperado por código genético")
plt.xlabel("Largo de ADN al azar (nt)")
plt.ylabel("Distancia L1 al esperado")
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.savefig(RESULTADOS_DIR / "nb_convergencia_aa_azar.png", dpi=180)
plt.show()

df_conv

Se observa que la distancia L1 cae aproximadamente como $1/\sqrt{N}$
(esperable para un proceso multinomial). Para secuencias de 20 000 nt o
más la distancia ya es del orden de 1-2 %, lo que se puede considerar una
aproximación "suficiente". Para 50 nt la distancia es del orden del 30 %,
es decir, todavía dominada por el ruido de muestreo.

### Ejercicio 4.1.d - Frecuencia observada vs número de codones por aminoácido

El código genético estándar tiene **61 codones que codifican aminoácidos**
(más 3 codones de stop). El número de codones por aminoácido va de **1**
(Met, Trp) a **6** (Leu, Ser, Arg). Si traducimos ADN al azar, la
frecuencia de cada aminoácido debería ser proporcional al número de
codones que lo codifican: por ejemplo, Leu con 6 codones aparece ~6 veces
más que Met con 1.

In [ ]:
codones_por_aa = Counter(v for v in CODIGO_GENETICO.values() if v != "*")
total_codones = sum(codones_por_aa.values())  # 61

tabla_codones = pd.DataFrame(
    [(aa, codones_por_aa.get(aa, 0), codones_por_aa.get(aa, 0) / total_codones) for aa in AMINOACIDOS],
    columns=["símbolo", "n_codones", "frecuencia_esperada"],
)

# Comparamos con una traducción de ADN al azar grande (60 000 nt) para reducir ruido
adn_grande = generar_adn(60000, semilla=42)
aa_grande = traducir_adn(adn_grande).replace("*", "")
freq_obs = frecuencias(aa_grande, AMINOACIDOS).rename(columns={"frecuencia": "frecuencia_observada"})

codones_vs_freq = tabla_codones.merge(freq_obs[["símbolo", "frecuencia_observada"]], on="símbolo")

ax = codones_vs_freq.sort_values("n_codones", ascending=False).plot(
    x="símbolo",
    y=["frecuencia_esperada", "frecuencia_observada"],
    kind="bar",
    figsize=(11, 4),
)
ax.set_title("Frecuencia esperada (n_codones / 61) vs observada en ADN al azar traducido")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.savefig(RESULTADOS_DIR / "nb_codones_vs_frecuencia.png", dpi=180)
plt.show()

codones_vs_freq.sort_values("n_codones", ascending=False)

Las dos series casi coinciden, lo que confirma que para una secuencia
al azar con composición de bases uniforme la frecuencia de aminoácidos
queda determinada por la **redundancia** del código genético. La
distribución no es uniforme (1/20 = 5 %): está claramente sesgada hacia
los aminoácidos con más codones (L, S, R) y empobrecida en Met y Trp,
que sólo tienen un codón.

## Objetivo 4.2 - Estadísticas de proteínas

Para este punto usé una tabla local descargada de UniProt con **500 proteínas revisadas de *Escherichia coli* K-12** (`organism_id:83333 AND reviewed:true`). La tabla conserva accession, longitud y nombre de proteína, suficiente para analizar tamaños.


In [ ]:
ecoli = pd.read_csv(DATA_DIR / "ecoli_k12_uniprot_reviewed_lengths.tsv", sep="\t")
tamanios_proteinas = ecoli["Length"].to_numpy()

media = tamanios_proteinas.mean()
percentil_15, percentil_85 = np.percentile(tamanios_proteinas, [15, 85])
print("Cantidad analizada:", len(tamanios_proteinas))
print("Media:", round(media, 1), "aa")
print("Rango central aproximado del 70%:", int(percentil_15), "a", int(percentil_85), "aa")

plt.figure(figsize=(8, 4))
plt.hist(tamanios_proteinas, bins=35, color="#4f8f6b", edgecolor="white")
plt.title("Histograma de tamaños proteicos de E. coli K-12")
plt.xlabel("Tamaño (aa)")
plt.ylabel("Cantidad")
plt.tight_layout()
plt.savefig(RESULTADOS_DIR / "nb_histograma_tamanios_proteicos.png", dpi=180)
plt.show()

ecoli.head()


La distribución se considera razonablemente estable cuando al aumentar el número de proteínas el histograma cambia poco y la media/rango central varían poco. Con 500 proteínas ya aparece una forma clara: la mayoría se concentra en tamaños intermedios y hay una cola de proteínas largas. Para una comparación más completa se podría descargar el proteoma completo y repetir el histograma con 100, 500, 1000 y todos los registros.


## Objetivo 4.3 - Distribución de aminoácidos en proteínas

Comparo una proteína aleatoria uniforme contra la mioglobina. En proteínas reales hay restricciones funcionales y estructurales, por lo que la distribución no necesariamente coincide con una proteína generada al azar.

In [ ]:
freq_prot_azar = frecuencias(proteina_azar, AMINOACIDOS)
comparacion_prot = freq_prot_azar.merge(freq_mioglobina, on="símbolo", suffixes=("_azar", "_mioglobina"))

ax = comparacion_prot.plot(
    x="símbolo",
    y=["frecuencia_azar", "frecuencia_mioglobina"],
    kind="bar",
    figsize=(10, 4),
)
ax.set_title("Proteína al azar vs mioglobina")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.savefig(RESULTADOS_DIR / "nb_comparacion_proteina_azar_mioglobina.png", dpi=180)
plt.show()

comparacion_prot[["símbolo", "frecuencia_azar", "frecuencia_mioglobina"]]

## Conclusión

El TP muestra dos ideas centrales: primero, que las bases de datos biológicas se pueden explorar manualmente con búsquedas avanzadas; segundo, que las mismas secuencias se pueden analizar programáticamente con Python/Biopython. Biopython evita reimplementar operaciones frecuentes como lectura de FASTA/GenBank/SwissProt, cálculo de GC, reversa complementaria, transcripción y traducción.